# Notes dbt & Analytics Engineering - Module 4 DE Zoomcamp

## Le rôle de l'Analytics Engineer

L'Analytics Engineer est né d'un **trou entre deux rôles** :

- **Data Engineer** → construit les pipelines, gère l'infra, ingère les données brutes
- **Data Analyst** → exploite les données pour répondre aux questions business

**L'Analytics Engineer** comble ce gap. Il :
- Transforme les données brutes en modèles analytiques propres et testés
- Applique des **pratiques d'ingénierie logicielle** (versioning, tests, documentation) au monde de la data
- Travaille principalement en SQL mais avec une approche structurée
- Comprend suffisamment le business pour modéliser correctement

> ⚠️ En PME (comme à La Réunion), le Data Engineer fait souvent aussi ce travail. Les deux rôles se confondent.

---

## Qu'est-ce que dbt ?

**dbt (Data Build Tool)** = outil qui permet de transformer des données dans un data warehouse en écrivant uniquement des `SELECT`.

### Le problème que dbt résout

Sans dbt, sur un projet de transformation SQL :
- Des dizaines de scripts SQL sans ordre d'exécution clair
- Pas de tests → si une colonne change en amont, tout casse silencieusement
- Pas de documentation → personne ne comprend la requête 6 mois plus tard
- Pas de versioning → impossible de revenir en arrière
- Les dépendances entre requêtes gérées à la main

dbt résout tous ces problèmes d'un coup.

### Comment ça fonctionne

Le principe fondamental : **tu écris uniquement des SELECT, dbt s'occupe de créer les tables/vues.**

```sql
-- models/staging/stg_trips.sql
SELECT
    vendor_id,
    pickup_datetime,
    total_amount
FROM {{ source('raw', 'green_taxi_trips') }}
WHERE vendor_id IS NOT NULL
```

dbt prend ce SELECT et exécute automatiquement un `CREATE TABLE AS SELECT ...` ou `CREATE VIEW AS SELECT ...` dans BigQuery.

---

## Structure d'un projet dbt

```
my_dbt_project/
├── dbt_project.yml          ← fichier le plus important
├── profiles.yml             ← connexion BDD (dbt Core seulement)
├── models/
│   ├── staging/             ← 1:1 avec les sources brutes
│   ├── intermediate/        ← transformations complexes non exposées
│   └── marts/               ← tables prêtes pour dashboards
│       ├── fct_trips.sql
│       └── dim_zones.sql
├── seeds/                   ← CSV à charger dans la BDD
├── macros/                  ← fonctions SQL réutilisables (comme Python)
├── tests/                   ← assertions en SQL
├── snapshots/               ← photos d'une table à un instant T
├── analyses/                ← SQL non exposé (rapports de qualité)
└── README.md                ← doc du projet
```

---

## Fichiers clés

### dbt_project.yml
Le fichier le plus important du projet. Il dit à dbt comment s'appelle le projet, quels sont les chemins des fichiers, et les matérialisations par défaut par couche.

```yaml
name: 'my_new_project'
version: '1.0.0'
profile: 'default'           # doit matcher ton profiles.yml (dbt Core)

model-paths: ["models"]
seed-paths: ["seeds"]
macro-paths: ["macros"]

models:
  my_new_project:
    staging:
      +materialized: view    # staging = vues par défaut
    marts:
      +materialized: table   # marts = tables par défaut
```

### profiles.yml (dbt Core uniquement)
Sur **dbt Cloud** → la connexion est configurée dans l'interface, pas de `profiles.yml`.
Sur **dbt Core** → le fichier est dans `~/.dbt/profiles.yml`.

```yaml
default:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: service-account
      project: mon-projet-gcp
      dataset: DBT_NY_Taxi
      location: europe-west2   # ⚠️ Doit matcher la région de tes données !
      keyfile: ./keys/my-creds.json
```

---

## Les couches de modélisation

```
Sources (raw) → Staging (stg_) → Intermediate (int_) → Marts (fct_ / dim_)
    brut           nettoyage         logique complexe       modèle final
```

### Staging
- **1:1 avec les sources brutes** (une table source = un fichier staging)
- Transformations minimales uniquement :
  - Cast des types de données
  - Renommage des colonnes
  - Filtre de base sur les nulls évidents
- Préfixe : `stg_`

```sql
-- models/staging/stg_green_tripdata.sql
SELECT
    vendorid AS vendor_id,
    CAST(lpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime,
    passenger_count,
    trip_distance,
    total_amount,
    'Green' AS service_type
FROM {{ source('raw', 'green_tripdata') }}
```

### Intermediate
- Tout ce qui n'est pas brut ET qu'on ne veut pas exposer directement
- Pas de conventions strictes, c'est pour le nettoyage lourd ou la logique complexe
- Préfixe : `int_`
- Exemple : union des tables green + yellow taxi

### Marts
- **Tables prêtes pour la consommation** (dashboards, analyses)
- Propriement modélisées et nettoyées
- Deux types :
  - **`fct_`** (fact tables) : événements, transactions — chaque ligne = un fait (un trajet, une vente)
  - **`dim_`** (dimension tables) : référentiels — clients, zones, dates

---

## Concepts dbt fondamentaux

### `source()` vs `ref()`

| Fonction | Usage | Lit depuis |
|----------|-------|-----------|
| `{{ source('schema', 'table') }}` | Données brutes externes | Table réelle dans la BDD |
| `{{ ref('nom_du_modele') }}` | Modèle dbt | Table/vue créée par dbt |

```sql
-- source() : lire les données brutes
FROM {{ source('raw', 'green_tripdata') }}

-- ref() : lire un modèle dbt
FROM {{ ref('stg_green_tripdata') }}
```

> ⚠️ **Bug classique** : `ref()` plante si tu n'as pas lancé `dbt run` au préalable. La preview marche car elle utilise la table brute, mais `ref()` a besoin que le modèle soit matérialisé.

### Les matérialisations

Façon dont dbt matérialise ton modèle dans la BDD :

| Matérialisation | Comportement | Usage |
|----------------|-------------|-------|
| `view` | Crée une vue SQL | Staging (pas de stockage) |
| `table` | Recrée la table complète à chaque run | Marts finaux |
| `incremental` | Ajoute uniquement les nouvelles lignes | Grandes tables de faits |
| `ephemeral` | Sous-requête CTE, n'existe pas dans la BDD | Logique intermédiaire légère |

Configurable dans le fichier ou directement dans le modèle :
```sql
{{ config(materialized='incremental') }}
SELECT ...
```

### Le DAG (Directed Acyclic Graph)
Comme dans Kestra : dbt construit automatiquement le graphe de dépendances via les `ref()`. Il exécute les modèles dans le bon ordre sans que tu aies à le gérer manuellement.

---

## Macros
Comme des fonctions Python, mais en Jinja+SQL. Permettent d'encapsuler de la logique réutilisable.

```sql
-- macros/get_payment_type_description.sql
{% macro get_payment_type_description(payment_type) %}
    CASE {{ payment_type }}
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        ELSE 'Unknown'
    END
{% endmacro %}

-- Utilisation dans un modèle
SELECT
    {{ get_payment_type_description('payment_type') }} AS payment_description
FROM {{ ref('stg_trips') }}
```

---

## Tests
Assertions en SQL — si la requête retourne plus de 0 lignes, `dbt build` échoue.

**Tests génériques (dans le schema.yml) :**
```yaml
models:
  - name: stg_green_tripdata
    columns:
      - name: trip_id
        tests:
          - unique
          - not_null
      - name: vendor_id
        tests:
          - accepted_values:
              values: [1, 2]
```

**Tests singuliers (fichier SQL dans /tests) :**
```sql
-- tests/assert_positive_trip_distance.sql
SELECT *
FROM {{ ref('stg_green_tripdata') }}
WHERE trip_distance < 0
-- Si cette requête retourne des lignes → le test échoue
```

---

## Seeds
Charger des fichiers CSV directement dans la BDD via dbt.

```bash
# Place le CSV dans seeds/
dbt seed
```

Utile pour les petites tables de référence (ex: codes de zones, types de véhicules).

> ⚠️ C'est une solution rapide. Mieux vaut corriger à la source quand c'est possible.

---

## Snapshots
Prend une "photo" d'une table à un instant T pour traquer l'historique d'une colonne qui s'écrase (Slowly Changing Dimensions de type 2).

```sql
-- snapshots/scd_dim_drivers.sql
{% snapshot scd_dim_drivers %}
    {{ config(
        target_schema='snapshots',
        unique_key='driver_id',
        strategy='timestamp',
        updated_at='updated_at',
    ) }}
    SELECT * FROM {{ source('raw', 'drivers') }}
{% endsnapshot %}
```

---

## Commandes essentielles

```bash
dbt init               # Créer un nouveau projet
dbt debug              # Vérifier la connexion
dbt run                # Exécuter tous les modèles
dbt run -s stg_trips   # Exécuter un seul modèle
dbt test               # Lancer tous les tests
dbt build              # run + test ensemble
dbt seed               # Charger les CSV (seeds)
dbt snapshot           # Exécuter les snapshots
dbt docs generate      # Générer la documentation
dbt docs serve         # Servir la doc localement
dbt clean              # Supprimer les dossiers target/ et dbt_packages/
```

---

## Modélisation dimensionnelle (Kimball)

### Fact tables vs Dimension tables
- **Fact table** = événements mesurables (transactions, trajets, ventes). Chaque ligne = un fait.
- **Dimension table** = contexte descriptif (qui, quoi, où). Sert à enrichir les faits.

### Grain
La **première décision** quand tu conçois une fact table : quel est le grain ?
- "Une ligne = un trajet" → grain trajet
- "Une ligne = un mois par zone" → grain mensuel/zone

### Méthode pour nommer les modèles

Le nom du modèle contient toute la logique :
```
fct_monthly_zone_revenue
 ^      ^      ^      ^
 |      |      |      mesure (revenu)
 |      |      dimension (zone)
 |      grain temporel (mensuel)
 type (fact)
```

### Conformed Dimensions
Plusieurs fact tables peuvent partager les mêmes dimensions :
```
              dim_commune
             /     |     \
            /      |      \
fct_ventes  |  fct_dpe    fct_risques
            \      |      /
             \     |     /
              dim_date
```
Permet les analyses croisées entre fact tables via les dimensions communes.

---

## Bugs classiques (vécus en pratique)

### 1. Région BigQuery
```
Error: Dataset was not found in location US
```
**Cause** : dbt cherche en région US mais ton dataset est en `europe-west2`.
**Fix** : Dans `profiles.yml`, ajouter `location: europe-west2`. Toutes les ressources doivent être dans la même région.

### 2. `ref()` qui plante
```
Error: Table not found: stg_green_tripdata
```
**Cause** : Tu utilises `ref()` mais n'as jamais lancé `dbt run`. Le modèle n'est pas matérialisé.
**Fix** : `dbt run` avant tout. `source()` lit les données brutes directement, `ref()` a besoin que le modèle existe.

### 3. Clé primaire composite et collisions
```sql
-- ❌ Dangereux : vendor_id=1, trip_id=23 → "123" == vendor_id=12, trip_id=3
CONCAT(vendor_id, trip_id)

-- ✅ Séparateur obligatoire
CONCAT(vendor_id, '-', trip_id)
```

---

## Déduplication avec ROW_NUMBER()

Pattern classique pour dédupliquer une table :

```sql
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER(
            PARTITION BY vendor_id, pickup_datetime, dropoff_datetime
            ORDER BY pickup_datetime
        ) AS row_num
    FROM {{ source('raw', 'green_tripdata') }}
)
SELECT * FROM ranked WHERE row_num = 1
```

> En pratique : 293 014 doublons identifiés dans les données NYC Taxi, réduits à 0 avec ce pattern.

---

## Récapitulatif Final

### Architecture dbt
```
Sources (BDD raw)
    ↓ source()
Staging (stg_) → vues, nettoyage minimal
    ↓ ref()
Intermediate (int_) → transformations complexes
    ↓ ref()
Marts → fct_ (faits) + dim_ (dimensions)
    ↓
Dashboards / BI tools
```

### Points clés à retenir

- **SELECT only** : jamais de `CREATE TABLE` ou `INSERT`, juste des `SELECT`
- **`source()`** pour les données brutes, **`ref()`** pour les modèles dbt
- **DAG automatique** : dbt gère l'ordre d'exécution via les `ref()`
- **`dbt run` avant `dbt test`** — ou utiliser `dbt build` pour tout faire d'un coup
- **Région BDD** : toutes les ressources dans la même région, toujours
- **Grain d'abord** : définir le grain avant de coder une fact table

### Ta stack complète à ce stade
```
Infrastructure (Terraform)
    ↓
Containerisation (Docker)
    ↓
Orchestration (Kestra)
    ↓
Data Warehouse (BigQuery)
    ↓
Transformation (dbt)    ← Module 4
    ↓
Visualisation (à venir)
```

---

*Notes du Module 4 - DataTalks Club DE Zoomcamp*